Q1. You are given a large NumPy array of size 5000000 ini8alized with random
values. Compute the following element-wise opera8on: f(x)=x2+3x+5, for
each element in the array and convert it into a CUDA kernel using Numba.
Compare performance difference of CPU with GPU.
a. Modify the kernel to use float32 and float64

In [9]:
import numpy as np
import time
from numba import cuda, njit

N = 5_000_000


def compute_cpu(x_arr):
    return x_arr**2 + 3*x_arr + 5


@njit(parallel=True)
def compute_cpu_jit(x_arr):
    out = np.empty_like(x_arr)
    for idx in range(x_arr.shape[0]):
        val = x_arr[idx]
        out[idx] = val*val + 3*val + 5
    return out


@cuda.jit
def kernel_f32(x_arr, out_arr):
    idx = cuda.grid(1)
    if idx < x_arr.shape[0]:
        val = x_arr[idx]
        out_arr[idx] = val*val + 3.0*val + 5.0


@cuda.jit
def kernel_f64(x_arr, out_arr):
    idx = cuda.grid(1)
    if idx < x_arr.shape[0]:
        val = x_arr[idx]
        out_arr[idx] = val*val + 3.0*val + 5.0


def exec_gpu(fn, host_arr, label):
    dev_in  = cuda.to_device(host_arr)
    dev_out = cuda.device_array(N, dtype=host_arr.dtype)

    tpb = 256
    bpg = (N + tpb - 1) // tpb

    fn[bpg, tpb](dev_in, dev_out)
    cuda.synchronize()

    t_start = time.perf_counter()
    fn[bpg, tpb](dev_in, dev_out)
    cuda.synchronize()
    t_end = time.perf_counter()

    res = dev_out.copy_to_host()
    print(f"  GPU ({label:>7}): {(t_end - t_start)*1000:.3f} ms")
    return res, t_end - t_start


def main():
    data32 = np.random.uniform(-100, 100, N).astype(np.float32)
    data64 = data32.astype(np.float64)

    t0 = time.perf_counter()
    out_cpu = compute_cpu(data64)
    t1 = time.perf_counter()
    cpu_dur = t1 - t0
    print(f"\n  CPU (NumPy)  : {cpu_dur*1000:.3f} ms")

    gpu_flag = cuda.is_available()
    print(f"\n  CUDA GPU available: {gpu_flag}")

    print()
    _, t32 = exec_gpu(kernel_f32, data32, "float32")
    _, t64 = exec_gpu(kernel_f64, data64, "float64")

    print(f"  CPU / GPU float32 : {cpu_dur/t32:.1f}x faster")
    print(f"  CPU / GPU float64 : {cpu_dur/t64:.1f}x faster")


if __name__ == "__main__":
    main()


  CPU (NumPy)  : 52.028 ms

  CUDA GPU available: True

  GPU (float32): 0.686 ms
  GPU (float64): 0.445 ms
  CPU / GPU float32 : 75.8x faster
  CPU / GPU float64 : 116.8x faster


Q2. Implement and benchmark a 1-D histogram computa8on for 1 million
random values in Python using Numba. Compare different approaches (pure
Python, NumPy, and Numba-accelerated) and analyze performance and
correctness. (Ref: hSps://numba.pydata.org/numbaexamples/examples/density_es8ma8on/histogram/results.html )

In [10]:
import numpy as np
import time
from numba import njit

arr = np.random.uniform(0, 100, 1_000_000)
edges = np.linspace(0, 100, 101)


def hist_py(x, e):
    cnt = [0] * (len(e) - 1)
    for v in x:
        for k in range(len(e) - 1):
            if e[k] <= v < e[k + 1]:
                cnt[k] += 1
                break
    return cnt


def hist_np(x, e):
    cnt, _ = np.histogram(x, e)
    return cnt


@njit
def hist_nb(x, e):
    cnt = np.zeros(len(e) - 1, dtype=np.int64)
    for i in range(len(x)):
        v = x[i]
        for j in range(len(e) - 1):
            if e[j] <= v < e[j + 1]:
                cnt[j] += 1
                break
    return cnt


t0 = time.time()
out_py = hist_py(arr, edges)
t1 = time.time()
py_dur = t1 - t0
print(f"Pure Python : {py_dur:.3f}s")

t0 = time.time()
out_np = hist_np(arr, edges)
t1 = time.time()
np_dur = t1 - t0
print(f"NumPy       : {np_dur:.4f}s")

hist_nb(arr[:100], edges)

t0 = time.time()
out_nb = hist_nb(arr, edges)
t1 = time.time()
nb_dur = t1 - t0
print(f"Numba       : {nb_dur:.4f}s")

print(f"\nPython vs Numba speedup : {py_dur/nb_dur:.1f}x")
print(f"NumPy  vs Numba speedup : {np_dur/nb_dur:.1f}x")

print(f"\ncorrectness:")

print(f"\nNumPy  vs Numba  : {np.allclose(out_np, out_nb)}")
print(f"Python vs NumPy  : {out_py[:5] == list(out_np[:5])}")

Pure Python : 14.460s
NumPy       : 0.0099s
Numba       : 0.0595s

Python vs Numba speedup : 242.8x
NumPy  vs Numba speedup : 0.2x

correctness:

NumPy  vs Numba  : True
Python vs NumPy  : True


Q3. Write a function monte_carlo_pi(nsamples) that estimates the value of π by
generating random x, y coordinates between 0 and 1 and checking if they fall
inside a unit circle (x2 + y2 < 1).

a. Implement the func8on in pure Python first and later create a Numba
version.

b. Program a script to compare the execu8on 8me for 5 million samples.
Report the Speedup Factor (Python Time / Numba Time).

c. Why does the very first execu8on of the Numba func8on take slightly
longer than the second execu8on?

In [3]:
import random
import time
import numpy as np
from numba import njit


def pi_py(n):
    cnt = 0
    for _ in range(n):
        a = random.uniform(0, 1)
        b = random.uniform(0, 1)
        if a*a + b*b < 1.0:
            cnt += 1
    return 4.0 * cnt / n


@njit
def pi_nb(n):
    cnt = 0
    for _ in range(n):
        a = np.random.uniform(0, 1)
        b = np.random.uniform(0, 1)
        if a*a + b*b < 1.0:
            cnt += 1
    return 4.0 * cnt / n


def measure(fn, n):
    t0 = time.time()
    val = fn(n)
    t1 = time.time()
    return val, t1 - t0


M = 5_000_000

res_py, t_py   = measure(pi_py, M)
res_nb1, t_nb1 = measure(pi_nb, M)
res_nb2, t_nb2 = measure(pi_nb, M)

print(f"Pure Python   : π ≈ {res_py:.5f}  | {t_py:.3f}s")
print(f"Numba 1st run : π ≈ {res_nb1:.5f}  | {t_nb1:.3f}s")
print(f"Numba 2nd run : π ≈ {res_nb2:.5f}  | {t_nb2:.3f}s")

sp = t_py / t_nb2
print(f"\nSpeedup = {sp:.1f}x faster with Numba")

Pure Python   : π ≈ 3.14181  | 2.468s
Numba 1st run : π ≈ 3.14046  | 0.192s
Numba 2nd run : π ≈ 3.14208  | 0.057s

Speedup = 43.2x faster with Numba


Q4. You have a 1D NumPy array representing pixel intensities (values 0–255). You
need to increase the brightness of every pixel by 20%, but ensure no value
exceeds 255.

a. Write a function adjust_brightness(pixel_value) using the @vectorize
decorator.

b. Apply this function to an array of 10 million random integers.

c. Change the decorator to @vectorize(['int64(int64)'], target='parallel').
Measure the time difference when the work is automatically split
across your CPU cores.

d. What happens if you try to pass a list instead of a NumPy array to this
function?

In [4]:
import numpy as np
import time
from numba import vectorize


@vectorize
def brighten(x):
    y = x * 1.2
    if y > 255:
        return 255
    return y


@vectorize(['int64(int64)'], target='parallel')
def brighten_par(x):
    y = x * 1.2
    if y > 255:
        return 255
    return y


arr = np.random.randint(0, 256, 10_000_000, dtype=np.int64)

t0 = time.time()
out1 = brighten(arr)
t1 = time.time() - t0

brighten_par(arr[:10])
t0 = time.time()
out2 = brighten_par(arr)
t2 = time.time() - t0

print(f"Basic    : {t1:.3f}s")
print(f"Parallel : {t2:.3f}s")
print(f"Speedup  : {t1/t2:.1f}x")
print(f"Sample   : {arr[:4]} → {out2[:4]}")

lst = [100, 200, 50, 255, 30]
res = brighten_par(lst)

print(f"List input  : {lst}")
print(f"Output      : {res}")

Basic    : 0.087s
Parallel : 0.021s
Speedup  : 4.2x
Sample   : [132  56 196  44] → [158  67 235  52]
List input  : [100, 200, 50, 255, 30]
Output      : [120 240  60 255  36]


Q5. Write Python code to generate synthetic training data of 100,000 samples,
10 features and binary labels {-1, +1}. Implement binary logistic regression
using the mathema8cal formula for gradient descent:

a. Using standard NumPy (without Numba)

b. Using Numba JIT acceleraIon

c. Compare correctness and performance.

In [5]:
import numpy as np
import time
from numba import njit

np.random.seed(42)
A = np.random.randn(100000, 10).astype(np.float64)
B = np.where(np.random.rand(100000) > 0.5, 1.0, -1.0).astype(np.float64)

eta = 0.01
steps = 100


def fit_np(a, b):
    w = np.zeros(10)
    c = 0.0
    for _ in range(steps):
        s   = a @ w + c
        p   = 1 / (1 + np.exp(-b * s))
        d   = (p - 1) * b
        w  -= eta * (a.T @ d) / len(b)
        c  -= eta * d.mean()
    return w, c


@njit
def fit_nb(a, b):
    w = np.zeros(10)
    c = 0.0
    for _ in range(steps):
        s   = a @ w + c
        p   = 1 / (1 + np.exp(-b * s))
        d   = (p - 1) * b
        w  -= eta * (a.T @ d) / len(b)
        c  -= eta * d.mean()
    return w, c


def score(a, b, w, c):
    return np.mean(np.sign(a @ w + c) == b) * 100


t0 = time.time()
w_np, c_np = fit_np(A, B)
t1 = time.time()
t_np = t1 - t0

fit_nb(A[:10], B[:10])

t0 = time.time()
w_nb, c_nb = fit_nb(A, B)
t1 = time.time()
t_nb = t1 - t0

print(f"NumPy  : {t_np:.3f}s")
print(f"Numba  : {t_nb:.3f}s")
print(f"Speedup: {t_np/t_nb:.1f}x")

NumPy  : 0.246s
Numba  : 0.399s
Speedup: 0.6x


Q6. Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [6]:
%%writefile matrix_add.py

import numpy as np
from numba import cuda

@cuda.jit
def matrix_add_kernel(A, B, C):
    row, col = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

N = 1024
A = np.random.randint(0, 100, (N, N)).astype(np.float32)
B = np.random.randint(0, 100, (N, N)).astype(np.float32)

A_gpu = cuda.to_device(A)
B_gpu = cuda.to_device(B)
C_gpu = cuda.device_array((N, N), dtype=np.float32)

threads = (16, 16)
blocks  = (64, 64)

matrix_add_kernel[blocks, threads](A_gpu, B_gpu, C_gpu)
cuda.synchronize()

C = C_gpu.copy_to_host()

print("Result C[0][0] :", C[0][0])
print("Expected       :", A[0][0] + B[0][0])
print("Correct        :", np.allclose(A + B, C))

Writing matrix_add.py


In [7]:
!python matrix_add.py

Result C[0][0] : 121.0
Expected       : 121.0
Correct        : True
